In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_clean = pd.read_csv('cleaned_aviation_data.csv')

# Separate small and large airplanes using 20-passenger threshold
small_planes = df_clean[df_clean['Total.Passengers'] <= 20]
large_planes = df_clean[df_clean['Total.Passengers'] > 20]

In [ ]:
# Lowest mean injury fraction by make
small_makes_safe = small_planes.groupby('Make')['Fatal_Serious_Fraction'].mean().sort_values().head(15)
large_makes_safe = large_planes.groupby('Make')['Fatal_Serious_Fraction'].mean().sort_values().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=small_makes_safe.values, y=small_makes_safe.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Small Airplanes: Top 15 Safest Makes')
axes[0].set_xlabel('Mean Fatal/Serious Injury Fraction')

sns.barplot(x=large_makes_safe.values, y=large_makes_safe.index, ax=axes[1], palette='Greens_r')
axes[1].set_title('Large Airplanes: Top 15 Safest Makes')
axes[1].set_xlabel('Mean Fatal/Serious Injury Fraction')
plt.tight_layout()
plt.show()

# Violinplot for small makes
top10_small = small_makes_safe.head(10).index
plt.figure(figsize=(14, 6))
sns.violinplot(data=small_planes[small_planes['Make'].isin(top10_small)], x='Make', y='Fatal_Serious_Fraction', inner='quartile')
plt.title('Injury Rate Distribution: Top 10 Safest Small Makes')
plt.xticks(rotation=45)
plt.show()

# Stripplot for large makes
top10_large = large_makes_safe.head(10).index
plt.figure(figsize=(14, 6))
sns.stripplot(data=large_planes[large_planes['Make'].isin(top10_large)], x='Make', y='Fatal_Serious_Fraction', alpha=0.5, jitter=True)
plt.title('Injury Rate Distribution: Top 10 Safest Large Makes')
plt.xticks(rotation=45)
plt.show()

In [ ]:
small_dest = small_planes.groupby('Make')['is_destroyed'].mean().sort_values().head(15)
large_dest = large_planes.groupby('Make')['is_destroyed'].mean().sort_values().head(15)

print("--- Safest Small Makes by Destruction Rate ---")
print(small_dest.head(5))

print("\n--- Safest Large Makes by Destruction Rate ---")
print(large_dest.head(5))

In [ ]:
# Filter plane types with at least 10 samples for statistical robustness
model_counts = df_clean['Plane_Type'].value_counts()
robust_models = df_clean[df_clean['Plane_Type'].isin(model_counts[model_counts >= 10].index)]

# Large plane models
large_robust = robust_models[robust_models['Total.Passengers'] > 20]
large_model_safe = large_robust.groupby('Plane_Type')['Fatal_Serious_Fraction'].mean().sort_values().head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=large_model_safe.values, y=large_model_safe.index, palette='viridis')
plt.title('Top 10 Safest Large Plane Models (Min. 10 Incidents)')
plt.xlabel('Mean Fatal/Serious Fraction')
plt.show()

# Small plane models
small_robust = robust_models[(robust_models['Total.Passengers'] <= 20) & (robust_models['Make'].isin(top10_small))]
small_model_safe = small_robust.groupby('Plane_Type')['Fatal_Serious_Fraction'].mean().sort_values().head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=small_model_safe.values, y=small_model_safe.index, palette='magma')
plt.title('Top 10 Safest Small Plane Models (Min. 10 Incidents)')
plt.xlabel('Mean Fatal/Serious Fraction')
plt.show()

In [ ]:
# Factor 1: Weather Condition
plt.figure(figsize=(8, 5))
sns.barplot(data=df_clean, x='Weather.Condition', y='Fatal_Serious_Fraction', palette='coolwarm')
plt.title('Impact of Weather Condition on Injury Fraction')
plt.show()

# Factor 2: Phase of Flight
phase_df = df_clean[df_clean['Broad.phase.of.flight'] != 'Unknown']
phase_order = phase_df.groupby('Broad.phase.of.flight')['is_destroyed'].mean().sort_values(ascending=False).index

plt.figure(figsize=(12, 5))
sns.barplot(data=phase_df, x='Broad.phase.of.flight', y='is_destroyed', order=phase_order, palette='Reds_r')
plt.title('Aircraft Destruction Rate by Phase of Flight')
plt.xticks(rotation=45)
plt.show()